In [ ]:
import visualizer
visualizer.start_server()


  EVOLANGDIFF CONTINUOUS MANIFOLD OBSERVER RUNNING
  Open your browser at: http://localhost:8008



----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 49178)
Traceback (most recent call last):
  File "C:\Users\chita\miniconda3\envs\py\lib\socketserver.py", line 683, in process_request_thread
    self.finish_request(request, client_address)
  File "C:\Users\chita\miniconda3\envs\py\lib\socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "C:\Users\chita\miniconda3\envs\py\lib\socketserver.py", line 747, in __init__
    self.handle()
  File "C:\Users\chita\miniconda3\envs\py\lib\http\server.py", line 435, in handle
    self.handle_one_request()
  File "C:\Users\chita\miniconda3\envs\py\lib\http\server.py", line 423, in handle_one_request
    method()
  File "C:\Users\chita\Desktop\Python\ideas\EvoLangDiff\visualizer.py", line 225, in do_POST
    self._send_binary(STATE.get_grid_state_bin())
  File "C:\Users\chita\Desktop\Python\ideas\EvoLangDiff\visualizer.py", line 184,

In [1]:
"""
test_phonetics.py — Concrete quantitative benchmark for EvoLangDiff phonetics.
Tests: (1) Epoch-0 emergence from empty inventory, (2) 200-step stability & inventory size,
(3) Environmental adaptation (Highland, Arid Desert, Dense Forest, Open Steppe).
"""

from types import SimpleNamespace
import numpy as np

from lexicon import LexiconGenerator
from phonetics import (
    sound_change_step,
    inventory_size,
    cluster_contrastive_phonemes,
)


def summarize_lexicon(words, label: str):
    v, c = cluster_contrastive_phonemes(words)
    sz = inventory_size(words, min_share=0.01)
    sample_forms = [w.form for w in words[:8]]
    print(f"\n[{label}]")
    print(f"  Active size (share>=1%): V={sz['vowels']} C={sz['consonants']} NonPulm={sz['nonpulmonic']}")
    print(f"  Reported Vowels ({len(v)}):     {' '.join(v)}")
    print(f"  Reported Consonants ({len(c)}): {' '.join(c)}")
    print(f"  Sample words: {' '.join(sample_forms)}")


def run_benchmark():
    print("=== 1. PROTO-LANGUAGE EMERGENCE (NO INITIAL INVENTORY) ===")
    core_v = {'a', 'i', 'u', 'e', 'o'}
    core_c = {'p', 'b', 't', 'd', 'k', 'ɡ', 'm', 'n', 'l', 'r', 's'}
    v_hits, c_hits = [], []

    for seed in range(5):
        rng = np.random.default_rng(100 + seed)
        words = LexiconGenerator.populate_proto_lexicon(population=0.5, rng=rng)
        v, c = cluster_contrastive_phonemes(words)
        v_hits.append(len(set(v) & core_v))
        c_hits.append(len(set(c) & core_c))
        if seed == 0:
            summarize_lexicon(words, f"Seed {seed} Proto (37 roots)")

    print(f"\n  Avg core vowels in {sorted(core_v)}: {np.mean(v_hits):.1f} / 5")
    print(f"  Avg core consonants in {sorted(core_c)}: {np.mean(c_hits):.1f} / 11")

    print("\n=== 2. 200-STEP AUTONOMOUS EVOLUTION (TEMPERATE SEA LEVEL) ===")
    rng = np.random.default_rng(42)
    env_temp = SimpleNamespace(altitude=0.1, humidity=0.55, temperature=0.55, vegetation=0.30, roughness=0.10, population=1.0)
    prof = SimpleNamespace(synthesis_index=0.50, syllable_complexity=0.50, potency=1.0, velocity=1.0)
    words = LexiconGenerator.populate_proto_lexicon(population=0.5, rng=rng)

    accepted = 0
    for step in range(1, 201):
        res = sound_change_step(words, rng=rng, T=0.05, env=env_temp, profile=prof)
        if res["accepted"]:
            accepted += 1
        if step in (50, 100, 200):
            sz = inventory_size(words, min_share=0.01)
            print(f"  Step {step:3d}: accept_rate={accepted/step:.2f} | V={sz['vowels']} C={sz['consonants']} Total={sz['total']}")
    summarize_lexicon(words, "Temperate @ 200 steps")

    print("\n=== 3. ECOLOGICAL DIVERGENCE (150 STEPS EACH) ===")
    biomes = {
        "Highland (alt=0.95)": SimpleNamespace(altitude=0.95, humidity=0.40, temperature=0.30, vegetation=0.15, roughness=0.60, population=0.5),
        "Arid Desert (hum=0.08, temp=0.90)": SimpleNamespace(altitude=0.15, humidity=0.08, temperature=0.90, vegetation=0.05, roughness=0.20, population=0.4),
        "Open Steppe (veg=0.05, rough=0.05)": SimpleNamespace(altitude=0.15, humidity=0.35, temperature=0.60, vegetation=0.05, roughness=0.05, population=0.5),
        "Dense Forest (veg=0.90, rough=0.45)": SimpleNamespace(altitude=0.10, humidity=0.85, temperature=0.70, vegetation=0.90, roughness=0.45, population=0.6),
    }
    base_words = LexiconGenerator.populate_proto_lexicon(population=0.5, rng=np.random.default_rng(7))
    for name, b_env in biomes.items():
        b_words = [w.copy() for w in base_words]
        b_rng = np.random.default_rng(77)
        for _ in range(150):
            sound_change_step(b_words, rng=b_rng, T=0.05, env=b_env, profile=prof)
        summarize_lexicon(b_words, name)


if __name__ == "__main__":
    run_benchmark()

=== 1. PROTO-LANGUAGE EMERGENCE (NO INITIAL INVENTORY) ===

[Seed 0 Proto (37 roots)]
  Active size (share>=1%): V=5 C=10 NonPulm=0
  Reported Vowels (3):     ɑ a ə
  Reported Consonants (9): ɾ t d l l̥ n m ŋ b
  Sample words: l̥ɑ nɑ ɾə mɑ ɾa ɾɑt ɾɑ tɑ

  Avg core vowels in ['a', 'e', 'i', 'o', 'u']: 0.8 / 5
  Avg core consonants in ['b', 'd', 'k', 'l', 'm', 'n', 'p', 'r', 's', 't', 'ɡ']: 4.4 / 11

=== 2. 200-STEP AUTONOMOUS EVOLUTION (TEMPERATE SEA LEVEL) ===
  Step  50: accept_rate=0.40 | V=5 C=7 Total=12
  Step 100: accept_rate=0.46 | V=4 C=6 Total=10
  Step 200: accept_rate=0.44 | V=3 C=4 Total=7

[Temperate @ 200 steps]
  Active size (share>=1%): V=3 C=4 NonPulm=0
  Reported Vowels (2):     æ ɒ
  Reported Consonants (4): k ɹ n̥ p
  Sample words: ɹæ ɹæn̥ kɒ pæ n̥æɹ ɹɒ kæ n̥ɒ

=== 3. ECOLOGICAL DIVERGENCE (150 STEPS EACH) ===

[Highland (alt=0.95)]
  Active size (share>=1%): V=5 C=5 NonPulm=0
  Reported Vowels (5):     ɞ ɯ ɑ a ɤ
  Reported Consonants (4): l̥ n m t
  Sample words: l̥